# Blueprint Reader: train the symbol detector on CubiCasa5K

Run on a free GPU (Colab: Runtime > Change runtime type > T4 GPU; or Kaggle with GPU on).

What this does: downloads CubiCasa5K (5.5 GB), converts its SVG annotations to YOLO boxes, lets you **look at the converted boxes**, fine-tunes a YOLO model, and scores it on the held-out test split.

The numbers this prints are the ones you may quote. Do not quote numbers you did not produce.

In [ ]:
!nvidia-smi -L
# Replace with your own repository once you have pushed the project to GitHub.
!git clone https://github.com/YOUR_USERNAME/blueprint-reader.git
%cd blueprint-reader
!pip -q install -e ".[train]"

## 1. Download the dataset

CubiCasa5K is licensed CC BY-NC-SA 4.0 (non-commercial). If the link has moved, get it from https://zenodo.org/records/2613548.

In [ ]:
!mkdir -p data
!wget -c -q --show-progress -O data/cubicasa5k.zip "https://zenodo.org/records/2613548/files/cubicasa5k.zip?download=1"
!unzip -q -n data/cubicasa5k.zip -d data
!ls data data/cubicasa5k | head -20

## 2. Convert annotations and inspect them

If `ls` above shows a different folder name, change `--src`. `train.txt`, `val.txt` and `test.txt` must be directly inside it.

In [ ]:
!python scripts/prepare_cubicasa.py --src data/cubicasa5k --out data/yolo --visualize 30

**Stop and look.** Every door box should cover the door leaf and its swing arc; fixtures should be boxed tightly. If door boxes look wrong, re-run the cell above with `--opening-mode opening` and compare. Also read the class counts printed above: very rare classes will score poorly, which is worth knowing before training.

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('data/yolo/preview/*.jpg'))[:8]:
    display(Image(p, width=700))

## 3. Quick sanity run (a few minutes)

Two epochs just to confirm everything works end to end before spending hours.

In [ ]:
!python scripts/train_detector.py --data data/yolo/data.yaml --epochs 2 --name sanity --out results/sanity.json

## 4. Full training run

This takes hours on a T4. Free Colab sessions can disconnect, so mount Drive first and copy the weights out when it finishes (cell below). Kaggle notebooks allow longer sessions and are the safer choice for this run.

Try at least two settings and keep both results, e.g. `yolo11n.pt` vs `yolo11s.pt`, or `--imgsz 640` vs `1024`. A comparison you ran yourself is far more convincing than a single number.

In [ ]:
!python scripts/train_detector.py --data data/yolo/data.yaml --model yolo11s.pt --epochs 60 --imgsz 1024 --batch 8 --name cubicasa_s_1024

In [ ]:
import json
print(json.dumps(json.load(open('results/detector_test.json')), indent=2))

## 5. Save the weights and results

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/blueprint-reader
!cp runs/detector/cubicasa_s_1024/weights/best.pt /content/drive/MyDrive/blueprint-reader/detector.pt
!cp -r results /content/drive/MyDrive/blueprint-reader/

## 6. Look at predictions, especially the mistakes

Open the validation plots Ultralytics wrote (confusion matrix, PR curves, `val_batch*_pred.jpg`). Write down which classes fail and why you think they fail. That list is your error analysis and the most useful thing to discuss in an interview.

In [ ]:
import glob
from IPython.display import Image, display
for p in sorted(glob.glob('runs/detector/cubicasa_s_1024_test/*.jpg'))[:4] + sorted(glob.glob('runs/detector/cubicasa_s_1024_test/*.png'))[:3]:
    print(p); display(Image(p, width=800))